# Coarse-Grained Animal Classification

This notebook performs coarse-grained classification across 10 animal classes using:
- provided image features,
- handcrafted image features,
- pretrained ResNet18 embeddings,
- and several supervised classification models.

The workflow compares feature representations, compares classification models using stratified 5-fold cross-validation, tunes the best model, and evaluates it on a holdout validation set.

In [ ]:
# library 
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = Path("../data/task1")
OUTPUT_DIR = Path("../results/task1")
CACHE_DIR = Path("../cache/task1")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "task1_predictions.csv"

RANDOM_STATE = 42

In [ ]:
DATA_DIR = Path("../data/task1")
OUTPUT_DIR = Path("../results/task1")
CACHE_DIR = Path("../cache/task1")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "task1_predictions.csv"

RANDOM_STATE = 42

In [ ]:
def engineer_image_features(data_dir: Path, metadata: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    
    """
    create additional features directly from images, including: 
    - RGB statistics
    - grayscale statistics 
    - color contrast
    - centre vs border brightness
    - local grid brightness
    """

    # process each image
    for _, row in metadata.iterrows():
        image_path = data_dir / row["image_path"]

        # convert image to RGB and resize to 64x64 
        img = Image.open(image_path).convert("RGB").resize((64, 64))

        # convert image to numpy array and normalise pixel values 
        arr = np.asarray(img).astype(np.float32) / 255.0

        # split RGB channels 
        r,g,b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

        # create grayscale image 
        gray = 0.299 * r + 0.587 * g + 0.114 * b 

        feature_dict = {"image_id": row["image_id"]}

        # RGB statistical features
        for name, channel in zip(["r", "g", "b"], [r, g, b]):
            feature_dict[f"{name}_mean"] = float(channel.mean())
            feature_dict[f"{name}_std"] = float(channel.std())

            # quantiles capture colour distribution
            feature_dict[f"{name}_q25"] = float(np.quantile(channel, 0.25))
            feature_dict[f"{name}_q50"] = float(np.quantile(channel, 0.50))
            feature_dict[f"{name}_q75"] = float(np.quantile(channel, 0.75))

        # Color contrast features 
        # red-green difference 
        rg = r - g

        # yellow-blue difference 
        yb = 0.5 * (r + g) - b 

        feature_dict["rg_mean_abs"] = float(np.mean(np.abs(rg)))
        feature_dict["yb_mean_abs"] = float(np.mean(np.abs(yb)))

        feature_dict["rg_std"] = float(np.std(rg))
        feature_dict["yb_std"] = float(np.std(yb))

        # Grayscale features 
        feature_dict["gray_mean"] = float(gray.mean())
        feature_dict["gray_std"] = float(gray.std())

        # Centre vs border brightness
        # extract centre region 
        centre = gray[16:48, 16:48]

        # create border mask 
        border_mask = np.ones_like(gray, dtype=bool)
        border_mask[16:48, 16:48] = False
        border = gray[border_mask]

        feature_dict["centre_mean"] = float(centre.mean())
        feature_dict["border_mean"] = float(border.mean())

        # difference between object centre and background 
        feature_dict["centre_border_diff"] = float(centre.mean() - border.mean())

        # Local brightness and grid features 
        # divide image into 4x4 regions 
        cell = 16
        for i in range(4):
            for j in range(4):
                patch = gray[
                    i * cell:(i + 1) * cell,
                    j * cell:(j + 1) * cell
                ]

                feature_dict[f"grid_gray_{i}_{j}"] = float(patch.mean())

        rows.append(feature_dict)
    return pd.DataFrame(rows)

In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# remove final classifier, so model outputs 512-dimensional features
resnet.fc = nn.Identity()

resnet = resnet.to(device)
resnet.eval()

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class ImageFeatureDataset(Dataset):
    def __init__(self, metadata, data_dir):
        self.metadata = metadata.reset_index(drop=True)
        self.data_dir = Path(data_dir)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        image_path = self.data_dir / row["image_path"]
        image = Image.open(image_path).convert("RGB")
        image = resnet_transform(image)

        return image, row["image_id"]

In [ ]:
def extract_resnet_features(metadata, data_dir, batch_size=64):
    dataset = ImageFeatureDataset(metadata, data_dir)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    all_features = []
    all_ids = []

    with torch.no_grad():
        for images, image_ids in loader:
            images = images.to(device)

            features = resnet(images)
            features = features.cpu().numpy()

            all_features.append(features)
            all_ids.extend(image_ids)

    all_features = np.vstack(all_features)

    feature_df = pd.DataFrame(
        all_features,
        columns=[f"resnet_{i}" for i in range(all_features.shape[1])]
    )

    feature_df.insert(0, "image_id", all_ids)

    return feature_df

In [ ]:
train_meta = pd.read_csv(DATA_DIR / "train_metadata.csv")
test_meta = pd.read_csv(DATA_DIR / "test_metadata.csv")

all_meta = pd.concat(
    [
        train_meta[["image_id", "image_path"]],
        test_meta[["image_id", "image_path"]]
    ],
    ignore_index=True
)

resnet_cache_path = data_dir / "resnet18_features_task1.csv"

if resnet_cache_path.exists():
    resnet_features = pd.read_csv(resnet_cache_path)
else:
    resnet_features = extract_resnet_features(
        all_meta,
        data_dir,
        batch_size=64
    )

    resnet_features.to_csv(
        resnet_cache_path,
        index=False
    )

resnet_features.head()